EXP 11

In [1]:
!pip install -q sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Documents
sentences = [
    "Python is a programming language",
    "Java is used for software development",
    "Machine learning is a part of AI",
    "Python is easy to learn"
]

# Generate embeddings
embeddings = model.encode(sentences)

# Search query
query = "I want to learn Python"
query_embedding = model.encode([query])

# Calculate similarity
scores = cosine_similarity(query_embedding, embeddings)[0]

# Display results
for i in range(len(sentences)):
    print(sentences[i], ":", round(scores[i], 3))

# Find the most similar sentence
best_match = scores.argmax()
print("\nBest Match:", sentences[best_match])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Python is a programming language : 0.745
Java is used for software development : 0.186
Machine learning is a part of AI : 0.161
Python is easy to learn : 0.82

Best Match: Python is easy to learn


EXP 12

In [3]:
!pip install -q sentence-transformers faiss-cpu

In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Documents
documents = [
    "Python is a programming language",
    "Java is an object oriented language",
    "AI makes machines intelligent",
    "Football is a popular sport"
]

# Create embeddings
vectors = model.encode(documents).astype('float32')

# Normalize vectors for cosine similarity
faiss.normalize_L2(vectors)

# Create FAISS database
dimension = vectors.shape[1]
database = faiss.IndexFlatIP(dimension)
database.add(vectors)

# Search
query = "Tell me about artificial intelligence"
query_vector = model.encode([query]).astype('float32')
faiss.normalize_L2(query_vector)

scores, indices = database.search(query_vector, k=2)

# Display results
print("Search Results:")
for i in range(2):
    print(documents[indices[0][i]],
          "Score:", round(float(scores[0][i]), 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Search Results:
AI makes machines intelligent Score: 0.703
Python is a programming language Score: 0.352


EXP 13

In [5]:
!pip install -q sentence-transformers transformers sentencepiece

In [8]:
!pip install -q transformers==4.46.3 huggingface-hub==0.36.2 sentence-transformers sentencepiece

In [1]:
import transformers

print(transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.46.3


In [2]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

answer = generator(
    "What is Python?",
    max_new_tokens=40
)

print(answer[0]["generated_text"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Python (disambiguation) Python is a syllable of arithmetic.


EXP 14

In [3]:
!pip install -q langchain-community langchain-core langchain-text-splitters faiss-cpu sentence-transformers transformers

In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# Computer Science knowledge
text = """
Python is a high-level programming language.
An operating system manages computer hardware and software.
A database stores and organizes information.
Machine learning allows computers to learn from data.
A computer network connects computers to share information.
"""

# Create documents
documents = [Document(page_content=text)]

# Split text into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)
chunks = splitter.split_documents(documents)

# Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector database
db = FAISS.from_documents(chunks, embeddings)

# Load question-answering model
qa = pipeline(
    "question-answering",
    model="distilbert/distilbert-base-cased-distilled-squad"
)

# Chatbot loop
while True:
    question = input("\nAsk a computer science question (type exit to stop): ")

    if question.lower() == "exit":
        print("Chatbot stopped.")
        break

    # Retrieve relevant documents
    results = db.similarity_search(question, k=2)
    context = " ".join(doc.page_content for doc in results)

    # Generate answer from context
    answer = qa(question=question, context=context)

    print("Chatbot:", answer["answer"])

/tmp/ipykernel_7140/98034328.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/tmp/ipykernel_7140/98034328.py:27: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Ask a computer science question (type exit to stop): What is python
Chatbot: a high-level programming language

Ask a computer science question (type exit to stop): exit
Chatbot stopped.


EXP 15

In [5]:
!pip install -q sentence-transformers faiss-cpu transformers

In [7]:
!pip install -q sentence-transformers faiss-cpu transformers

import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Create a document inside the program
text = """
Python is a high-level programming language.
An operating system manages computer hardware and software.
A database stores and organizes information.
Machine learning allows computers to learn from data.
A computer network connects computers to share information.
"""

# Split document into chunks
chunks = [line.strip() for line in text.split("\n") if line.strip()]

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
vectors = model.encode(chunks).astype("float32")
faiss.normalize_L2(vectors)

# Create FAISS vector database
db = faiss.IndexFlatIP(vectors.shape[1])
db.add(vectors)

# Load question-answering model
qa = pipeline(
    "question-answering",
    model="distilbert/distilbert-base-cased-distilled-squad"
)

# AI assistant
while True:
    question = input("\nAsk a question (type exit to stop): ")

    if question.lower() == "exit":
        print("Assistant stopped.")
        break

    # Find relevant information
    query_vector = model.encode([question]).astype("float32")
    faiss.normalize_L2(query_vector)

    scores, indices = db.search(query_vector, k=2)

    context = " ".join(chunks[i] for i in indices[0] if i >= 0)

    # Answer the question
    result = qa(question=question, context=context)

    print("AI Assistant:", result["answer"])


Ask a question (type exit to stop): What is python?
AI Assistant: Python

Ask a question (type exit to stop):  What does an operating system do?
AI Assistant: manages computer hardware and software

Ask a question (type exit to stop): exit
Assistant stopped.
